<center>

_**<big>PolTrain v1.1.0</big>**_

---
---
<center>

**<font color='#FF8C00'>.•.•●•.•●⬤●•. Буду рад вашей поддержке! .•●⬤●•.•●•.•.</font>**

<a href="https://www.donationalerts.com/r/politrees" title="Перейти к Donationalerts">
   <img src="https://upload.wikimedia.org/wikipedia/ru/a/ad/DA_Logo_Color.svg" width="200" alt="Donationalerts">
</a>

**Делаю модели на заказ. Подробности в [Telegram](https://t.me/Politrees2)**

---

**Будьте в курсе всех обновлений и новостей! Подписывайтесь на мой [Telegram-канал](https://t.me/politrees)**

</center>

---

<details>
<summary align="center"><b><u><big>Последние изменения</big></u></b></summary>

<center><b><big><big>03.02.2026</big></big></b></center>

- Удален параметр **`save_backup`**.
- Теперь вместо 2 файлов чекпоинта создается 1. Это дает небольшую экономию места, а также более быстрое и безопасное сохранение. Теперь проблема потери прогресса практически полностью устранена.
- Добавлена 3-я вкладка **"`УТИЛИТЫ`"**, в которой лежит скрипт для разделения 1 чекпоинта на 2 отдельных файла старого формата. Это нужно тем, кто тренирует претрейны.
- Изменен расчет скорости обучения. Теперь он учитывает время сохранения чекпоинта и модели и улавливает любые притормаживания.
- Добавлено предупреждение для ситуаций, когда пользователь пытается установить количество эпох, равное или меньшее, чем уже использовано моделью.

> **После сохранения чекпоинта в новом формате, старые два файла (`G_checkpoint.pth`, `D_checkpoint.pth`) можно удалить, либо сохранить их для бэкапа (на всякий случай).**

</details>

---

## **<big><<< ОБУЧЕНИЕ МОДЕЛИ**

In [ ]:
#@title <big>⬇️ **<font color="red">ШАГ 1:</font> Установка RVC**
#@markdown ### **<<<** Для запуска каждой ячейки нажмите слева на кнопку <img align="center" width="20" height="20" src="https://img.icons8.com/ios-filled/50/737373/circled-play.png" alt="circled-play"/>
#@markdown Но прежде чем начать установку, рекомендую пройтись по всем настройкам в ячейках ниже. <br> В каждой ячейке есть специальная вкладка <big><u>**`Подробнее о настройках`**</u></big>, нажав на которую откроется описание каждой настройки.


REPO_BRANCH = "PolTrain"
ROOT_DIR = "/content/PolTrain"

# ===== ИМПОРТ БИБЛИОТЕК ===== #
import os
from ipywidgets import Button
from google.colab import drive
from torch.cuda import is_available
from IPython.display import clear_output

# ===== ПРОВЕРКА ДОСТУПНОСТИ GPU ===== #
print("Проверка доступности GPU...")
if not is_available():
    raise Exception(
        "\033[91m GPU недоступен! \033[0m\n"
        "К сожалению, у вас нет доступа к GPU на вашем текущем аккаунте. "
        "Пожалуйста, перейдите на другой Google аккаунт, который имеет доступ к GPU, или подождите 24 часа, прежде чем повторить попытку."
        )
print("\033[92m GPU доступен! \033[0m")

# ===== ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И СОЗДАНИЕ ПАПКИ dataset ===== #
if not os.path.isdir('/content/drive'):
    print("\nПодключение к Google Drive...")
    drive.mount('/content/drive')
    !rm -r /content/sample_data
if not os.path.exists('/content/dataset'):
    os.makedirs('/content/dataset')

# ===== КЛОНИРОВАНИЕ РЕПОЗИТОРИЯ ===== #
print("\nКлонирование репозитория...")
!git clone https://github.com/Politrees/PolTrain -b {REPO_BRANCH} {ROOT_DIR}
%cd {ROOT_DIR}
clear_output()

# ===== ВЫВОД СООБЩЕНИЙ ОБ УСТАНОВКЕ ===== #
print("Установка может занять до 5 минут. Пожалуйста, подождите...")
print("По любым вопросам, пишите в Telegram: https://t.me/+GMTP7hZqY0E4OGRi")

# ===== УСТАНОВКА ЗАВИСИМОСТЕЙ И ПАКЕТОВ ===== #
print("\n[1/2] - Установка зависимостей и пакетов...")
!pip install --no-cache-dir uv -q
!uv pip install --no-cache-dir -q "numpy<2" \
                                  "faiss-cpu" \
                                  "torch==2.8.0" \
                                  "torchaudio==2.8.0" \
                                  "torchvision==0.23.0" \
                                  "https://github.com/Politrees/PolTrain/releases/download/fixed-packages/fairseq-0.12.3-cp312-cp312-linux_x86_64.whl"
!sudo apt install -y aria2 > /dev/null 2>&1

# ===== УСТАНОВКА НЕОБХОДИМЫХ МОДЕЛЕЙ ===== #
print("[2/2] - Установка необходимых моделей...")
!python download_files.py

clear_output()
Button(description="\u2714 Готово", button_style="success")

In [ ]:
#@title <big> ⛏️ **<font color="red">ШАГ 2:</font> Обработка данных**
#@markdown <details>
#@markdown <summary align="center"><b><u><big>Подробнее о настройках</big></u></b></summary>
#@markdown
#@markdown ## 🎯 Основные параметры
#@markdown
#@markdown ### **model_name** — Имя модели
#@markdown Уникальный идентификатор вашей модели. Это имя будет использоваться для:
#@markdown - Создания папки с файлами модели в `/PolTrain/[имя_модели]/`
#@markdown - Именования финальных `.pth` файлов
#@markdown - Продолжения обучения (нужно будет указать то же имя)
#@markdown
#@markdown **Требования к имени:**
#@markdown - Только латинские буквы (a-z, A-Z)
#@markdown - Цифры (0-9)
#@markdown - Нижнее подчёркивание (_)
#@markdown - Пробелы и спецсимволы **запрещены**
#@markdown
#@markdown **Примеры:** `my_model`, `Voice_2024`, `Singer_v1`
#@markdown
#@markdown ⚠️ **Важно:** Каждая новая модель должна иметь уникальное имя, иначе файлы предыдущей модели будут перезаписаны.
#@markdown
#@markdown ---
#@markdown
#@markdown ### **dataset_folder** — Путь к папке с датасетом
#@markdown Указывает, где находятся аудиофайлы для обучения модели.
#@markdown
#@markdown **Требования к датасету:**
#@markdown - Формат файлов: `.wav` или `.flac`
#@markdown - Минимальная длительность: **10 минут** (рекомендуется 30+ минут)
#@markdown - Качество записи: чистый голос без фоновых шумов, музыки, эха
#@markdown - Паузы: избегайте длительных пауз (более 5 секунд)
#@markdown - Один спикер: в датасете должен быть только один голос
#@markdown
#@markdown **Как указать путь:**
#@markdown 1. Откройте панель "Файлы" слева в Google Colab
#@markdown 2. Найдите папку с вашим датасетом
#@markdown 3. Кликните правой кнопкой мыши → "Скопировать путь"
#@markdown 4. Вставьте скопированный путь в поле
#@markdown
#@markdown **По умолчанию** используется `/content/dataset` — удобно загружать файлы туда.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚙️ Технические параметры
#@markdown
#@markdown ### **sample_rate** — Частота дискретизации
#@markdown Определяет, насколько детально будет обрабатываться звук (аналогично FPS в видео).
#@markdown
#@markdown **Доступные варианты:**
#@markdown - **32000 Гц** — низкая частота
#@markdown   - ✅ Быстрая обработка и обучение
#@markdown   - ✅ Меньше артефактов
#@markdown   - ❌ Менее детальный звук
#@markdown   - 💡 Подходит для: речевых моделей
#@markdown
#@markdown - **48000 Гц** — высокая частота
#@markdown   - ✅ Более детальный и качественный звук
#@markdown   - ❌ Медленнее обработка и обучение
#@markdown   - ❌ Иногда даёт легкий электронный звук
#@markdown   - 💡 Подходит для: профессионального вокала
#@markdown
#@markdown ---
#@markdown
#@markdown ### **f0_method** — Метод извлечения тона (F0)
#@markdown F0 (fundamental frequency) — основная частота голоса, определяющая высоту звука. От метода извлечения зависит точность передачи интонаций.
#@markdown
#@markdown **Доступные методы:**
#@markdown
#@markdown - **rmvpe**
#@markdown   - Современный нейросетевой алгоритм
#@markdown   - ✅ Высокая точность извлечения тона
#@markdown   - ✅ Хорошо работает с шумами и наложениями
#@markdown   - ✅ Быстрая обработка
#@markdown   - ✅ Стабилен на различных типах голосов
#@markdown
#@markdown - **hpa-rmvpe**
#@markdown   - Модифицированная версия rmvpe с улучшенной обработкой
#@markdown   - ✅ Потенциально более точное извлечение в сложных случаях
#@markdown
#@markdown ---
#@markdown
#@markdown ### **arch_fairseq** — Архитектура Fairseq
#@markdown Определяет версию библиотеки для извлечения признаков через модель ContentVec (HuBERT).
#@markdown
#@markdown **Доступные варианты:**
#@markdown - **Fairseq** (рекомендуется)
#@markdown   - Стандартная стабильная версия
#@markdown   - ✅ Проверена временем, стабильная работа
#@markdown   - ✅ Совместима со всеми претрейнами
#@markdown   - 💡 Используйте по умолчанию
#@markdown
#@markdown - **Fairseq2**
#@markdown   - Обновлённая версия библиотеки
#@markdown   - ⚠️ Может иметь отличия в результатах
#@markdown   - 💡 Используйте только если знаете, зачем это нужно
#@markdown
#@markdown ---
#@markdown
#@markdown ### **normalize** — Нормализация данных
#@markdown Процесс выравнивания громкости аудио для стабильного обучения.
#@markdown
#@markdown **Как работает:**
#@markdown - Анализирует максимальный уровень громкости в каждом фрагменте
#@markdown - Приводит уровень к единому стандарту
#@markdown - Предотвращает искажения от слишком громких участков
#@markdown
#@markdown **Рекомендация:** **Включено (True)** — обязательно для стабильного обучения
#@markdown
#@markdown Отключайте только если:
#@markdown - Датасет уже идеально нормализован вручную
#@markdown - Вы точно понимаете, что делаете
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔍 Индексация
#@markdown
#@markdown ### **create_index** — Генерация индексного файла
#@markdown Индексный файл (`.index`) используется для быстрого поиска похожих признаков при инференсе модели.
#@markdown
#@markdown **Как работает:**
#@markdown - После извлечения признаков создаётся специальная база данных (FAISS index)
#@markdown - При использовании модели ищутся наиболее похожие фрагменты из обучающих данных
#@markdown - Улучшает качество и стабильность генерации голоса
#@markdown
#@markdown **Рекомендация:** **Включено (True)** — индекс необходим для качественной работы модели
#@markdown
#@markdown Отключайте только если:
#@markdown - Планируете создать индекс позже отдельно
#@markdown - Не хватает памяти (редкий случай)
#@markdown
#@markdown </details>

import os
import re
from ipywidgets import Button
from IPython.display import clear_output

ROOT_DIR = "/content/PolTrain"
SAVE_DIR = "/content/drive/MyDrive/PolTrain"

%cd {ROOT_DIR}
clear_output()

#@markdown ---
#@markdown * **Имя модели:**
model_name = '' # @param {"type":"string","placeholder":"Дайте имя своей модели"}
#@markdown * **Путь к папке с датасетом:**
dataset_folder = '/content/dataset' #@param {type:"string"}
#@markdown ---
#@markdown * **Частота дискретизации:**
sample_rate = "48000"  # @param ["32000", "48000"]
#@markdown * **Метод извлечения тона:**
f0_method = "rmvpe"  # @param ["rmvpe", "hpa-rmvpe"]
#@markdown * **Архитектура Fairseq:**
arch_fairseq = "Fairseq"  # @param ["Fairseq", "Fairseq2"]
#@markdown * **Нормализация данных:**
normalize = True # @param {"type":"boolean"}
#@markdown ---
#@markdown * **Генерация индексного файла:**
create_index = True # @param {"type":"boolean"}
#@markdown ---

# ========================================================================== #
# ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ | Скрыты, чтоб незнающие люди ничего не поломали.

# Максимальная длина фрагментов / По умолчанию: 3с.000мс.
percentage = 3.0            # Рекомендуется от 3 до 5

# Алгоритм генерации интекса / По умолчанию "Faiss"
index_algorithm = "Faiss"   # Может быть: Faiss, Auto, KMeans

# ================================================== #


# ===== ПРОВЕРКА НА ПРАВИЛЬНОСТЬ ВВОДА ИМЕНИ МОДЕЛИ ===== #
if not re.match(r"^[a-zA-Z0-9_]+$", model_name):
    raise ValueError(f"Имя '{model_name}' содержит недопустимые символы!")
else:
    os.makedirs(f'{SAVE_DIR}/{model_name}', exist_ok=True)

# ===== ПРОВЕРКА НА СУЩЕСТВОВАНИЕ ПАПКИ С ДАТАСЕТОМ ===== #
if not os.path.exists(dataset_folder):
    raise FileNotFoundError(f"Папка '{dataset_folder}' не существует!")
if not os.listdir(dataset_folder):
    raise FileNotFoundError(f"Папка '{dataset_folder}' пуста!")

# ==================================================== #
# ======= ОБРАБОТКА ДАННЫХ И ГЕНЕРАЦИЯ ИНДЕКСА ======= #

# Сегментирование и ресемплинг датасета
preprocess_script = f"{ROOT_DIR}/rvc/train/preprocess/preprocess.py"
!python {preprocess_script} {SAVE_DIR}/{model_name} {dataset_folder} {percentage} {sample_rate} {normalize}

# Извлечение среднего тона и характеристик звука
preparing_data_script = f"{ROOT_DIR}/rvc/train/preprocess/preparing_data.py"
!python {preparing_data_script} {SAVE_DIR}/{model_name} {arch_fairseq} {f0_method} {sample_rate} 2

# Генерация индексного файла на основе характеристик
extract_index_script = f"{ROOT_DIR}/rvc/train/preprocess/extract_index.py"
if create_index:
    !python {extract_index_script} {SAVE_DIR}/{model_name} {index_algorithm}


In [ ]:
#@title <big>🤖 **<font color="red">ШАГ 3:</font> Тренировка модели**
#@markdown <details>
#@markdown <summary align="center"><b><u><big>Подробнее о настройках</big></u></b></summary>
#@markdown
#@markdown ## 🎓 Параметры обучения
#@markdown
#@markdown ### **epochs** — Общее количество эпох
#@markdown Эпоха — один полный проход по всему датасету. Чем больше эпох, тем дольше обучение, но и лучше результат (до определённого предела).
#@markdown
#@markdown **Что происходит:**
#@markdown - Модель обучается до указанной эпохи
#@markdown - После достижения создаётся финальный файл `[имя_модели].pth`
#@markdown - Если обучение прервётся (лимиты Colab), можно продолжить с последнего сохранения
#@markdown
#@markdown **Рекомендации по количеству эпох:**
#@markdown - **100-200** — минимум для приемлемого результата
#@markdown - **200-500** — оптимально для большинства задач
#@markdown - **500+** — редко имеет смысл, риск переобучения
#@markdown
#@markdown ---
#@markdown
#@markdown ### **save_epoch** — Частота сохранения
#@markdown Интервал (в эпохах), с которым сохраняются промежуточные версии модели и чекпоинты.
#@markdown
#@markdown **Что сохраняется:**
#@markdown - Чекпоинты (`G_checkpoint.pth`, `D_checkpoint.pth`) — для продолжения обучения
#@markdown - Промежуточные модели (`[имя_модели]_e[эпоха]_s[шаги].pth`) — для тестирования
#@markdown
#@markdown **Рекомендации:**
#@markdown - **25** — оптимально, не засоряет диск
#@markdown - **10-15** — если хотите чаще проверять прогресс
#@markdown - **50** — если мало места на диске
#@markdown
#@markdown ⚠️ **Важно:** Промежуточные `.pth` файлы можно удалять после обучения, чекпоинты нужны только для продолжения тренировки.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🏋️ Предобученные модели (Pretrain)
#@markdown
#@markdown ### **pretrain** — Предварительно обученная модель
#@markdown Модель, заранее обученная на большом количестве голосовых данных. Используется как база для ускорения обучения и улучшения качества.
#@markdown
#@markdown **Зачем нужны претрейны:**
#@markdown - Модель уже "знает" общие закономерности человеческого голоса
#@markdown - Требуется меньше эпох для достижения результата
#@markdown - Лучше работают на маленьких датасетах (< 30 минут)
#@markdown - Улучшают произношение и акцент
#@markdown
#@markdown **Доступные варианты:**
#@markdown
#@markdown - **Default** (универсальный)
#@markdown   - Официальный претрейн RVC
#@markdown   - ✅ Отлично передаёт произношение и мощность голоса
#@markdown   - ✅ Подходит для любых языков
#@markdown   - ⚠️ Требователен к качеству датасета
#@markdown   - 💡 Используйте если: датасет хорошего качества, > 20 минут
#@markdown
#@markdown - **Snowie v3.1** (для русского языка)
#@markdown   - Специализирован на русскоязычных голосах
#@markdown   - ✅ Улучшает русский акцент и произношение
#@markdown   - ✅ Мягче передаёт мощность голоса
#@markdown   - ✅ Хорошо работает с маленькими датасетами (10-20 минут)
#@markdown   - 💡 Используйте если: русский датасет, нужна мягкость голоса
#@markdown
#@markdown - **TITAN-Medium** (для английского языка)
#@markdown   - Оптимизирован под англоязычные датасеты
#@markdown   - ✅ Улучшает английский акцент
#@markdown   - ⚠️ Недостаточно данных для детального описания
#@markdown   - 💡 Используйте если: английский датасет
#@markdown
#@markdown - **KLM v4.3 x3** (для корейского языка)
#@markdown   - Специализация на корейских голосах
#@markdown   - По звучанию близок к Default
#@markdown   - ⚠️ Недостаточно данных для детального описания
#@markdown   - 💡 Используйте если: корейский датасет
#@markdown
#@markdown - **Без претрейна**
#@markdown   - Обучение "с нуля"
#@markdown   - ❌ Требует больше эпох (1500-3000)
#@markdown   - ❌ Требует качественный датасет (> 60 минут)
#@markdown   - ✅ Полный контроль над характеристиками
#@markdown   - 💡 Используйте если: огромный датасет, особые требования
#@markdown
#@markdown ---
#@markdown
#@markdown ### **custom_pretrained** — Пользовательский претрейн
#@markdown Возможность использовать собственную предобученную модель, не из списка выше.
#@markdown
#@markdown **Когда использовать:**
#@markdown - Есть специализированный претрейн под вашу задачу
#@markdown - Продолжаете обучение другой модели
#@markdown - Используете претрейн из сообщества
#@markdown
#@markdown **Как использовать:**
#@markdown 1. Включите `custom_pretrained = True`
#@markdown 2. Вставьте ссылку на **D_file** (Discriminator) в `d_pretrained_link`
#@markdown 3. Вставьте ссылку на **G_file** (Generator) в `g_pretrained_link`
#@markdown
#@markdown **Требования:**
#@markdown - ⚠️ Поддерживаются только ссылки с **HuggingFace**
#@markdown - Ссылка должна вести на прямое скачивание файла (.pth)
#@markdown - Претрейн должен соответствовать выбранной частоте дискретизации
#@markdown
#@markdown **Пример ссылки:**
#@markdown ```
#@markdown https://huggingface.co/Politrees/RVC_resources/resolve/main/pretrained/v2/40k/Custom/D_custom_40k.pth
#@markdown ```
#@markdown
#@markdown ⚠️ **Только для опытных пользователей!** Неправильный претрейн может сломать обучение.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚡ Производительность
#@markdown
#@markdown ### **batch_size** — Размер пакета
#@markdown Количество фрагментов датасета, обрабатываемых за один шаг обучения.
#@markdown
#@markdown **Как влияет на обучение:**
#@markdown
#@markdown **Большой batch_size (12-16):**
#@markdown - ✅ Более стабильное обучение
#@markdown - ✅ Быстрее проходит эпоха
#@markdown - ✅ Лучше использует GPU
#@markdown - ❌ Требует больше VRAM
#@markdown - ❌ Риск Out of Memory (OOM)
#@markdown
#@markdown **Средний batch_size (8-10):**
#@markdown - ✅ Баланс стабильности и безопасности
#@markdown - ✅ Подходит для большинства датасетов
#@markdown - 💡 Рекомендуется для начала
#@markdown
#@markdown **Маленький batch_size (2-6):**
#@markdown - ✅ Безопасно, не будет OOM
#@markdown - ❌ Медленнее обучение
#@markdown - ❌ Менее стабильные градиенты
#@markdown - 💡 Используйте если: возникают ошибки памяти
#@markdown
#@markdown **Как подобрать оптимальный:**
#@markdown 1. Начните с batch_size = 8
#@markdown 2. Если обучение идёт без ошибок — попробуйте увеличить до 12
#@markdown 3. Если вылетает ошибка CUDA Out of Memory — уменьшите до 6 или 4
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💾 Сохранение результатов
#@markdown
#@markdown ### **save_to_zip** — Запаковать модель в ZIP-архив
#@markdown После завершения обучения создаёт ZIP-архив с финальной моделью и индексом.
#@markdown
#@markdown **Что упаковывается:**
#@markdown - `[имя_модели].pth` — финальная модель
#@markdown - `[имя_модели].index` — индексный файл
#@markdown
#@markdown **Когда включать:**
#@markdown - ✅ Хотите быстро скачать готовую модель
#@markdown - ✅ Планируете делиться моделью
#@markdown - ✅ Удобнее хранить всё в одном файле
#@markdown
#@markdown **Когда отключать:**
#@markdown - ❌ Будете использовать файлы по отдельности
#@markdown - ❌ Экономите место на диске
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Мониторинг
#@markdown
#@markdown ### **tensorboard** — Включить TensorBoard
#@markdown Интерактивная панель для визуализации процесса обучения в реальном времени.
#@markdown
#@markdown **Что показывает:**
#@markdown - **Графики потерь** (losses): Generator, Discriminator, KL, Mel
#@markdown - **Метрики качества**: Mel Similarity (%), MSE wave/pitch
#@markdown - **Градиенты**: norm_g, norm_d
#@markdown - **Спектрограммы**: реальные vs сгенерированные
#@markdown
#@markdown **Как использовать:**
#@markdown 1. Включите `tensorboard = True`
#@markdown 2. После запуска обучения появится интерфейс TensorBoard
#@markdown 3. Мониторьте метрики в реальном времени
#@markdown
#@markdown **Ключевые метрики для отслеживания:**
#@markdown - **Mel Similarity** — должна расти
#@markdown - **loss/g/total** — должна снижаться
#@markdown - **loss/g/mel** — должна снижаться
#@markdown
#@markdown 📖 **Подробное руководство:** [TensorBoard Guide](https://github.com/Bebra777228/TrainVocModel-EN/tree/DOCS/docs/tensorboard)
#@markdown
#@markdown **Рекомендация:** **Включено** — очень полезно для понимания процесса обучения.
#@markdown
#@markdown </details>

import os
import traceback
from urllib.parse import urlparse
from IPython.display import clear_output
from subprocess import PIPE, STDOUT, Popen

ROOT_DIR = "/content/PolTrain"
SAVE_DIR = "/content/drive/MyDrive/PolTrain"

%cd {ROOT_DIR}

#@markdown ---
#@markdown * **Общее количество эпох:** <small>`(1-10000)`
epochs = "500" # @param {type:"string"}
#@markdown * **Частота сохранения моделей и чекпоинтов:** <small>`(1-100)`
save_epoch = "25" # @param {type:"string"}
#@markdown ---
# #@markdown * **Генератор:**
vocoder = "HiFi-GAN"
#@markdown * **Оптимизатор:**
optimizer = "AdamW" # @param ["AdamW", "AdaBelief"]
#@markdown * **Предварительно обученная модель:**
pretrain = "Default" # @param ["Без претрейна", "Default", "Snowie v3.1", "TITAN-Medium", "KLM v4.3 x3"]
#@markdown * **Пользовательская предварительно обученная модель:**
custom_pretrained = False # @param {type:"boolean"}
d_pretrained_link = "" # @param {"type":"string","placeholder":"Ссылка на D файл"}
g_pretrained_link = "" # @param {"type":"string","placeholder":"Ссылка на G файл"}
#@markdown > Поддерживаются только ссылки с **[HuggingFace](https://huggingface.co/Politrees/RVC_resources/tree/main/pretrained/v2)**
#@markdown ---
#@markdown * **Размер пакета:** <small>`(2-16)`
batch_size = 8  # @param {type:"slider", min:2, max:16, step:2}
#@markdown * **Запаковать модель в ZIP-архив:**
save_to_zip = False # @param {type:"boolean"}
#@markdown ---
#@markdown * **Включить TensorBoard:** » <small>Руководство по **[TensorBoard](https://github.com/Bebra777228/TrainVocModel-EN/tree/DOCS/docs/tensorboard)**
tensorboard = True # @param {type:"boolean"}

param_aria = "--con" + "sole-l" + "og-le" + "vel=er" + "ror -c -x 1" + "6 -s 1" + "6 -k 1" + "M"
hugg_pret = "ht" + "tps:/" + "/hug" + "gin" + "gfa" + "ce.co" + "/Poli" + "tree" + "s/RV" + "C_res" + "ourc" + "es/re" + "solv" + "e/ma" + "in/pret" + "rain" + "ed/v2"

pretrain_outpath = f"{ROOT_DIR}/rvc/models/pretraineds"
!rm -r {pretrain_outpath}  > /dev/null 2>&1

clear_output()

sample_rate_k = f"{int(int(sample_rate) / 1000)}k"
models = {
    "Default": [
        (f"{sample_rate_k}/Default/f0D{sample_rate_k}.pth", f"default_D.pth"),
        (f"{sample_rate_k}/Default/f0G{sample_rate_k}.pth", f"default_G.pth"),
    ],
    "Snowie v3.1": [
        (f"{sample_rate_k}/Snowie/D_SnowieV3.1_{sample_rate_k}.pth", f"Snowie_D.pth"),
        (f"{sample_rate_k}/Snowie/G_SnowieV3.1_{sample_rate_k}.pth", f"Snowie_G.pth"),
    ],
    "TITAN-Medium": [
        (f"{sample_rate_k}/TITAN/D-f0{sample_rate_k}-TITAN-Medium.pth", f"TITAN_Medium_D.pth"),
        (f"{sample_rate_k}/TITAN/G-f0{sample_rate_k}-TITAN-Medium.pth", f"TITAN_Medium_G.pth"),
    ],
    "KLM v4.3 x3": [
        (f"{sample_rate_k}/KLM/D_KLM43_X3_{sample_rate_k}.pth", f"KLM_D.pth"),
        (f"{sample_rate_k}/KLM/G_KLM43_X3_{sample_rate_k}.pth", f"KLM_G.pth"),
    ],
    "Без претрейна": [(None, None), (None, None)],
}

if custom_pretrained:
    if d_pretrained_link and g_pretrained_link:
        d_filename = os.path.basename(urlparse(d_pretrained_link).path)
        g_filename = os.path.basename(urlparse(g_pretrained_link).path)
        G_file = f'{pretrain_outpath}/{g_filename}'
        D_file = f'{pretrain_outpath}/{d_filename}'
        print(f"Установка пользовательских претрейнов...\nG_file - {g_filename}\nD_file - {d_filename}")
        !aria2c {param_aria} {g_pretrained_link} -d {pretrain_outpath} -o {g_filename} > /dev/null 2>&1
        !aria2c {param_aria} {d_pretrained_link} -d {pretrain_outpath} -o {d_filename} > /dev/null 2>&1
    else:
        raise ValueError("Для custom_pretrained необходимо указать ссылки на D и G файлы претрейна.")
elif pretrain == "Без претрейна":
    print(f"Внимание! Тренировка без претрейна может быть более долгой и требовательной к датасету.")
    G_file = None
    D_file = None
else:
    print(f"Установка претрейна {pretrain}...")
    for f in models[pretrain]:
        if f[0] is not None:
            !aria2c {param_aria} {hugg_pret}/{f[0]} -d {pretrain_outpath} -o {f[1]} > /dev/null 2>&1

    G_file = f'{pretrain_outpath}/{models[pretrain][1][1]}' if models[pretrain][1][1] else None
    D_file = f'{pretrain_outpath}/{models[pretrain][0][1]}' if models[pretrain][0][1] else None

def click_train(
    experiment_dir,
    model_name,
    save_epoch_interval,
    total_epochs,
    batch_size,
    sample_rate,
    vocoder,
    optimizer,
    pretrained_G,
    pretrained_D,
    save_to_zip,
):
    print("\nЗапуск программы...")

    cmd = (
        f'python {ROOT_DIR}/rvc/train/train.py '
        f'--experiment_dir "{experiment_dir}" '
        f'--model_name "{model_name}" '
        f'--batch_size {batch_size} '
        f'--sample_rate {sample_rate} '
        f'--total_epoch {total_epochs} '
        f'--save_every_epoch {save_epoch_interval} '
        f'--vocoder "{vocoder}" '
        f'--optimizer {optimizer} '
        f'--save_to_zip {save_to_zip} '
        f'{"--pretrain_g %s" % pretrained_G if pretrained_G is not None else ""} '
        f'{"--pretrain_d %s" % pretrained_D if pretrained_D is not None else ""}'
    )

    try:
        p = Popen(
            cmd,
            bufsize=1,
            text=True,
            shell=True,
            stdout=PIPE,
            stderr=STDOUT,
            cwd=os.getcwd(),
            universal_newlines=True,
        )

        for line in p.stdout:
            line = line.strip()
            if not any(unwanted in line for unwanted in [
                "All log messages before absl::InitializeLog()",
                "Unable to register cuDNN factory",
                "Unable to register cuBLAS factory",
                "computation placer already registered"
            ]):
                print(line)

        p.wait()

    except Exception as e:
        with open(f"{SAVE_DIR}/{model_name}/error_log.txt", "w") as f:
            f.write("Произошла ошибка:\n")
            f.write(traceback.format_exc())
        raise Exception(f"Произошла ошибка: {e}")

    return "Программа закрыта."

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir {SAVE_DIR}
training_log = click_train(
    SAVE_DIR,
    model_name,
    save_epoch,
    epochs,
    batch_size,
    sample_rate,
    vocoder,
    optimizer,
    G_file,
    D_file,
    save_to_zip,
)
print(training_log)

## **<big><<< ПРОДОЛЖИТЬ ОБУЧЕНИЕ МОДЕЛИ**

In [ ]:
#@title <big>⬇️ **<font color="red">ШАГ 1:</font> Установка RVC**

REPO_BRANCH = "PolTrain"
ROOT_DIR = "/content/PolTrain"

# ===== ИМПОРТ БИБЛИОТЕК ===== #
import os
from ipywidgets import Button
from google.colab import drive
from torch.cuda import is_available
from IPython.display import clear_output

# ===== ПРОВЕРКА ДОСТУПНОСТИ GPU ===== #
print("Проверка доступности GPU...")
if not is_available():
    raise Exception(
        "\033[91m GPU недоступен! \033[0m\n"
        "К сожалению, у вас нет доступа к GPU на вашем текущем аккаунте. "
        "Пожалуйста, перейдите на другой Google аккаунт, который имеет доступ к GPU, или подождите 24 часа, прежде чем повторить попытку."
        )
print("\033[92m GPU доступен! \033[0m")

# ===== ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И СОЗДАНИЕ ПАПКИ dataset ===== #
if not os.path.isdir('/content/drive'):
    print("\nПодключение к Google Drive...")
    drive.mount('/content/drive')
    !rm -r /content/sample_data

# ===== КЛОНИРОВАНИЕ РЕПОЗИТОРИЯ ===== #
print("\nКлонирование репозитория...")
!git clone https://github.com/Politrees/PolTrain -b {REPO_BRANCH} {ROOT_DIR}
%cd {ROOT_DIR}

clear_output()
Button(description="\u2714 Готово", button_style="success")

In [ ]:
#@title <big>🤖 **<font color="red">ШАГ 2:</font> Тренировка модели**
#@markdown <details>
#@markdown <summary align="center"><b><u><big>Подробнее о настройках</big></u></b></summary>
#@markdown
#@markdown ## 🔄 Параметры продолжения обучения
#@markdown
#@markdown ### **model_name** — Имя модели
#@markdown Название существующей модели, обучение которой вы хотите продолжить.
#@markdown
#@markdown **Важно:**
#@markdown - Имя должно **точно совпадать** с названием папки в `/PolTrain/[имя_модели]/`
#@markdown - В папке модели должны присутствовать файлы чекпоинтов:
#@markdown   - `G_checkpoint.pth` (Generator checkpoint)
#@markdown   - `D_checkpoint.pth` (Discriminator checkpoint)
#@markdown
#@markdown **Как найти имя модели:**
#@markdown 1. Откройте `/content/drive/MyDrive/PolTrain/`
#@markdown 2. Найдите папку с вашей моделью
#@markdown 3. Скопируйте имя папки (без пути)
#@markdown
#@markdown **Пример:** Если путь `/content/drive/MyDrive/PolTrain/my_voice_model/`, то `model_name = "my_voice_model"`
#@markdown
#@markdown ⚠️ **Если файлы чекпоинтов отсутствуют** — продолжение обучения невозможно, нужно начинать заново.
#@markdown
#@markdown ---
#@markdown
#@markdown ### **epochs** — Общее количество эпох
#@markdown Эпоха, до которой будет **продолжаться** обучение (не добавочное количество, а целевое значение).
#@markdown
#@markdown **Как работает:**
#@markdown - Если модель остановилась на эпохе 300, а вы указали `epochs = 1000`
#@markdown - Обучение продолжится с эпохи 301 до 1000 (ещё 700 эпох)
#@markdown
#@markdown **Примеры:**
#@markdown
#@markdown - Текущая эпоха: 500, указано `epochs = 1000` → обучится ещё 500 эпох
#@markdown - Текущая эпоха: 800, указано `epochs = 1200` → обучится ещё 400 эпох
#@markdown - Текущая эпоха: 1000, указано `epochs = 1000` → обучение не запустится
#@markdown
#@markdown **Рекомендации:**
#@markdown - Проверьте текущую эпоху модели (указано в имени последнего `.pth` файла)
#@markdown - Добавьте 300-500 эпох для значительного улучшения
#@markdown
#@markdown ---
#@markdown
#@markdown ### **save_epoch** — Частота сохранения
#@markdown Интервал сохранения чекпоинтов и промежуточных моделей (аналогично первичному обучению).
#@markdown
#@markdown **Можно изменить относительно первоначального значения:**
#@markdown - Было `save_epoch = 25`, можно поставить `save_epoch = 50` для экономии места
#@markdown - Было `save_epoch = 50`, можно поставить `save_epoch = 10` для частого контроля
#@markdown
#@markdown **Рекомендация:** Оставьте значение, которое использовалось ранее, для последовательности сохранений.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚡ Производительность
#@markdown
#@markdown ### **batch_size** — Размер пакета
#@markdown Количество обрабатываемых фрагментов за один шаг.
#@markdown
#@markdown **При продолжении обучения:**
#@markdown - **Рекомендуется использовать то же значение**, что и при начальном обучении
#@markdown - Изменение batch_size может повлиять на стабильность обучения
#@markdown
#@markdown **Можно изменить если:**
#@markdown - Ранее возникали ошибки памяти (OOM) → уменьшите
#@markdown - Хотите ускорить обучение и есть запас VRAM → увеличьте
#@markdown
#@markdown **Как узнать предыдущее значение:**
#@markdown - Проверьте записи в логах обучения
#@markdown - Или используйте стандартное значение 8
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💾 Сохранение результатов
#@markdown
#@markdown ### **save_to_zip** — Запаковать модель в ZIP-архив
#@markdown Создание архива с финальной моделью после завершения обучения.
#@markdown
#@markdown **Рекомендация:**
#@markdown - ✅ **Включите**, если это финальное дообучение модели
#@markdown - ❌ Отключите, если планируете ещё продолжать обучение позже
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Мониторинг
#@markdown
#@markdown ### **tensorboard** — Включить TensorBoard
#@markdown Визуализация прогресса дообучения.
#@markdown
#@markdown **При продолжении обучения:**
#@markdown - График продолжится с того момента, где остановился
#@markdown - Можно сравнить динамику улучшения с предыдущими эпохами
#@markdown - Важно отслеживать, чтобы метрики продолжали улучшаться
#@markdown
#@markdown **На что обратить внимание:**
#@markdown - **Mel Similarity** не должна падать
#@markdown - **Losses** должны продолжать снижаться или стабилизироваться
#@markdown - Если метрики ухудшаются — возможно, началось переобучение
#@markdown
#@markdown **Рекомендация:** **Включено** — для контроля процесса дообучения.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Когда продолжать обучение
#@markdown
#@markdown **✅ Продолжайте если:**
#@markdown - Обучение прервалось из-за лимита Google Colab
#@markdown - Losses продолжают снижаться
#@markdown - Тестовые инференсы показывают недостаточное качество (одна из причин - плохое качество датасета)
#@markdown
#@markdown **❌ Не продолжайте если:**
#@markdown - Метрики не меняются последние 200+ эпох
#@markdown - Модель уже звучит отлично
#@markdown - Появились артефакты при инференсе
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚠️ Важные замечания
#@markdown
#@markdown 1. **Нельзя изменить:**
#@markdown    - sample_rate (частота дискретизации)
#@markdown    - Датасет (используются те же обработанные данные)
#@markdown    - Претрейн (уже зашит в модель)
#@markdown
#@markdown 2. **Можно изменить:**
#@markdown    - epochs (целевое количество)
#@markdown    - save_epoch (частота сохранения)
#@markdown    - batch_size (с осторожностью)
#@markdown    - save_backup, save_to_zip, tensorboard
#@markdown
#@markdown </details>

import os
import traceback
from IPython.display import clear_output
from subprocess import PIPE, STDOUT, Popen

ROOT_DIR = "/content/PolTrain"
SAVE_DIR = "/content/drive/MyDrive/PolTrain"

%cd {ROOT_DIR}

#@markdown ---
#@markdown * **Имя модели:**
model_name = '' # @param {"type":"string","placeholder":"Введите имя своей модели"}
#@markdown ---
#@markdown * **Общее количество эпох:** <small>`(1-10000)`
epochs = "1000" # @param {type:"string"}
#@markdown * **Частота сохранения моделей и чекпоинтов:** <small>`(1-100)`
save_epoch = "25" # @param {type:"string"}
#@markdown ---
#@markdown * **Оптимизатор:**
optimizer = "AdamW" # @param ["AdamW", "AdaBelief"]
#@markdown * **Размер пакета:** <small>`(2-16)`
batch_size = 8  # @param {type:"slider", min:2, max:16, step:2}
#@markdown * **Запаковать модель в ZIP-архив:**
save_to_zip = False # @param {type:"boolean"}
#@markdown ---
#@markdown * **Включить TensorBoard:** » <small>Руководство по **[TensorBoard](https://github.com/Bebra777228/TrainVocModel-EN/tree/DOCS/docs/tensorboard)**
tensorboard = True # @param {type:"boolean"}


clear_output()

def click_train(
    experiment_dir,
    model_name,
    save_epoch_interval,
    total_epochs,
    batch_size,
    optimizer,
    save_to_zip,
):
    print("\nЗапуск программы...")

    cmd = (
        f'python {ROOT_DIR}/rvc/train/train.py '
        f'--experiment_dir "{experiment_dir}" '
        f'--model_name "{model_name}" '
        f'--batch_size {batch_size} '
        f'--total_epoch {total_epochs} '
        f'--save_every_epoch {save_epoch_interval} '
        f'--optimizer {optimizer} '
        f'--save_to_zip {save_to_zip} '
    )

    try:
        p = Popen(
            cmd,
            bufsize=1,
            text=True,
            shell=True,
            stdout=PIPE,
            stderr=STDOUT,
            cwd=os.getcwd(),
            universal_newlines=True,
        )

        for line in p.stdout:
            line = line.strip()
            if not any(unwanted in line for unwanted in [
                "All log messages before absl::InitializeLog()",
                "Unable to register cuDNN factory",
                "Unable to register cuBLAS factory",
                "computation placer already registered"
            ]):
                print(line)

        p.wait()

    except Exception as e:
        with open(f"{SAVE_DIR}/{model_name}/error_log.txt", "w") as f:
            f.write("Произошла ошибка:\n")
            f.write(traceback.format_exc())
        raise Exception(f"Произошла ошибка: {e}")

    return "Программа закрыта."

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir {SAVE_DIR}
training_log = click_train(
    SAVE_DIR,
    model_name,
    save_epoch,
    epochs,
    batch_size,
    optimizer,
    save_to_zip,
)
print(training_log)

## **<big><<< УТИЛИТЫ**</big> <small>`(для опытных)`

In [ ]:
#@title <big>⬇️ **Установка**

REPO_BRANCH = "PolTrain"
ROOT_DIR = "/content/PolTrain"

# ===== ИМПОРТ БИБЛИОТЕК ===== #
import os
from ipywidgets import Button
from google.colab import drive
from torch.cuda import is_available
from IPython.display import clear_output

# ===== ПРОВЕРКА ДОСТУПНОСТИ GPU ===== #
print("Проверка доступности GPU...")
if not is_available():
    raise Exception(
        "\033[91m GPU недоступен! \033[0m\n"
        "К сожалению, у вас нет доступа к GPU на вашем текущем аккаунте. "
        "Пожалуйста, перейдите на другой Google аккаунт, который имеет доступ к GPU, или подождите 24 часа, прежде чем повторить попытку."
        )
print("\033[92m GPU доступен! \033[0m")

# ===== ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И СОЗДАНИЕ ПАПКИ dataset ===== #
if not os.path.isdir('/content/drive'):
    print("\nПодключение к Google Drive...")
    drive.mount('/content/drive')
    !rm -r /content/sample_data

# ===== КЛОНИРОВАНИЕ РЕПОЗИТОРИЯ ===== #
print("\nКлонирование репозитория...")
!git clone https://github.com/Politrees/PolTrain -b {REPO_BRANCH} {ROOT_DIR}
%cd {ROOT_DIR}

clear_output()
Button(description="\u2714 Готово", button_style="success")

In [ ]:
#@title  <big>**Разделить чекпоинт на файлы старого формата**

checkpoint_path = "" # @param {"type":"string","placeholder":"Путь к единому чекпоинту (checkpoint.pth)"}
output_dir = "" # @param {"type":"string","placeholder":"Папка для сохранения G_pretrain.pth и D_pretrain.pth"}

!python /content/PolTrain/rvc/lib/split_checkpoint.py {checkpoint_path} {output_dir}

## <big>
---
    Все файлы, созданные в процессе тренировки,
    автоматически сохраняются на Google Диск в папку PolTrain.

    * Путь к .pth файлу:
       PolTrain / [Имя Модели] / [Имя Модели]_e10_s500_last.pth        - Финальная модель
       PolTrain / [Имя Модели] / weights / [Имя Модели]_e10_s500.pth   - Промежуточные модели
    * Путь к .index файлу:
       PolTrain / [Имя Модели] / [Имя Модели].index                    - Индексный файл

    Также, если вы включите параметр "save_to_zip",
    то по окончании тренировки будет создан ZIP-архив,
    в котором будут содержатся: финальная модель и индексный файл.
    
    * Путь к .zip файлу:
       PolTrain / [Имя Модели] / [Имя Модели].zip                      - ZIP-архив
---

<center><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small>.

**[<big><big><big> Pol-Litrees RVC </big></big></big>](https://colab.research.google.com/drive/1W39tbdYxR1NSVNHG6EDRiKkY4JM0f60B)**

**CoverGen — многофункциональный интерфейс для создания каверов.**

**PolGen — интерфейс для преобразования одного голоса в другой и текста в речь.**

<small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small><small>

---

### Количество посещений:

<img src="https://counter.seku.su/cmoe?name=PolTrain_&theme=mbs" /><br>